# Book separator model training

This notebook is a thin UI over the reusable training package. It works in Google Colab or local Jupyter. Use a GPU runtime in Colab before training.

In [ ]:
from pathlib import Path
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    PROJECT_ROOT = Path('/content/book-spine-detector/ml')
else:
    candidates = (Path.cwd(), Path.cwd() / 'ml', Path.cwd().parent)
    PROJECT_ROOT = next((p for p in candidates if (p / 'pyproject.toml').exists()), Path.cwd())
print('Colab:', IN_COLAB)
print('Project root:', PROJECT_ROOT)
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('Set PROJECT_ROOT to the cloned/uploaded project directory.')

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', f'{PROJECT_ROOT}[notebook]'])
os.chdir(PROJECT_ROOT)

## Prepare a Roboflow/YOLO export

Export a public dataset in YOLOv8 segmentation, YOLO OBB, or YOLO detection format. Set the source directory below. The converter draws the two long edges of every annotated book into a binary separator mask.

In [ ]:
from book_spine_ml.prepare_yolo import prepare_yolo_dataset

YOLO_EXPORT = Path('/content/public-book-spines') if IN_COLAB else PROJECT_ROOT / 'data/raw/public-book-spines'
PROCESSED_DATA = PROJECT_ROOT / 'data/processed/public_books'

# Uncomment after YOLO_EXPORT contains train/images, train/labels, valid/images, ...
# summary = prepare_yolo_dataset(YOLO_EXPORT, PROCESSED_DATA, line_width=5, prefix='spineseg_')
# summary

## Inspect converted targets
Always inspect masks before spending GPU time. The colored lines should follow the long sides of each visible book.

In [ ]:
from book_spine_ml.data import SeparatorDataset, preview_sample

dataset = SeparatorDataset(PROCESSED_DATA, 'train', image_size=640, augment=False)
print('Training samples:', len(dataset))
preview_sample(dataset, index=0)

## Train
Adjust `configs/baseline.yaml` first if necessary. Checkpoints and history are written after every epoch, so interrupted Colab sessions do not lose the best completed epoch.

In [ ]:
import torch
from book_spine_ml.train import train_from_config

print('CUDA available:', torch.cuda.is_available())
RESUME_FROM = None  # Example: PROJECT_ROOT / 'runs/separator-baseline/last.pt'
result = train_from_config(PROJECT_ROOT / 'configs/baseline.yaml', resume_path=RESUME_FROM)
result['checkpoint']

In [ ]:
import matplotlib.pyplot as plt

history = result['history']
plt.figure(figsize=(10, 4))
plt.plot([row['epoch'] for row in history], [row['train']['loss'] for row in history], label='train loss')
plt.plot([row['epoch'] for row in history], [row['val']['loss'] for row in history], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(True); plt.legend(); plt.show()

## Evaluate and export ONNX

In [ ]:
from book_spine_ml.evaluate import evaluate_checkpoint
from book_spine_ml.export_onnx import export_checkpoint

checkpoint = Path(result['checkpoint'])
metrics = evaluate_checkpoint(checkpoint, split='test')
print(metrics)
onnx_path = export_checkpoint(checkpoint, checkpoint.parent / 'separator_model.onnx')
print('Exported:', onnx_path)